# 03 — Funding Amount Prediction

This notebook trains and evaluates machine learning models to estimate funding amounts using features available at prediction time.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from src.data_loader import load_raw_csv
from src.preprocessing import preprocess
from src.features import prepare_features, create_time_split
from src.train import run_training
from src.evaluate import plot_actual_vs_predicted, plot_residuals, plot_model_comparison
from src.config import CLEANED_CSV

## 1. Load and Prepare Data

In [ ]:
if CLEANED_CSV.exists():
    df = pd.read_csv(CLEANED_CSV)
else:
    raw = load_raw_csv()
    df = preprocess(raw)

print(f"Dataset shape: {df.shape}")

## 2. Feature Preparation

In [ ]:
X, y = prepare_features(df)
print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nFeature columns: {list(X.columns)}")
print(f"\nTarget statistics:")
print(y.describe())

## 3. Time-Based Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = create_time_split(X, y, test_size=0.2)
print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")
print(f"\nTrain year range: {X_train['year'].min():.0f} - {X_train['year'].max():.0f}")
print(f"Test year range: {X_test['year'].min():.0f} - {X_test['year'].max():.0f}")

## 4. Model Training

In [ ]:
results, pipeline, preprocessor, best_name, paths = run_training(X_train, y_train, X_test, y_test)
print(f"\nBest model (by inner CV RMSLE): {best_name}")
print(f"\nModel paths: {paths}")

print("\nTarget skew is ~22, so RMSLE is the selection metric: raw-dollar")
print("metrics are dominated by the few billion-dollar rounds.")

## 5. Results Summary

In [ ]:
results_df = pd.DataFrame({
    'Model': list(results.keys()),
    'CV RMSLE': [results[k].get('cv_rmsle') for k in results],
    'Test RMSLE': [results[k]['rmsle'] for k in results],
    'MAE (USD)': [results[k]['mae'] for k in results],
    'RMSE (USD)': [results[k]['rmse'] for k in results],
    'R²': [results[k]['r2'] for k in results],
})
print(results_df.to_string(index=False))

print(f"\nSelected by inner CV (not test): {best_name}")
print(f"Tuned params: {results[best_name].get('best_params')}")

## 6. Model Comparison Visualization

In [ ]:
plot_model_comparison(results)
plt.savefig('../reports/model_comparison.png', dpi=150)
plt.show()

## 7. Actual vs Predicted (Best Model)

In [ ]:
best_pred = results[best_name]['predictions']
plot_actual_vs_predicted(y_test.values, best_pred, best_name)
plt.savefig(f'../reports/actual_vs_predicted_{best_name.replace(" ", "_").lower()}.png', dpi=150)
plt.show()

## 8. Residual Analysis

In [ ]:
plot_residuals(y_test.values, best_pred, best_name)
plt.savefig(f'../reports/residuals_{best_name.replace(" ", "_").lower()}.png', dpi=150)
plt.show()

## 9. Interpretation

Key observations:

1. **Baseline comparison:** The DummyRegressor provides a baseline. Any useful model should beat it.
2. **Time-based split:** Earlier events are used for training, later events for testing. This simulates real-world prediction.
3. **Log transformation:** The HistGradientBoosting model uses log1p transformation to handle skewed funding amounts.
4. **Feature importance:** See the explainability notebook for detailed feature analysis.
5. **Limitations:** Model estimates are not guarantees of funding. They reflect historical patterns only.